# Reproducing Xue, Buccino et al. (2022) from dandiset 000223

**Paper.** Xue X, Buccino AP, Saseendran Kumar S, Hierlemann A, Bartram J (2022). Inferring
monosynaptic connections from paired dendritic spine Ca2+ imaging and large-scale recording of
extracellular spiking. *Journal of Neural Engineering* 19(4), 046044.
[doi:10.1088/1741-2552/ac8765](https://doi.org/10.1088/1741-2552/ac8765). The claims file was
written from the accepted author manuscript in PubMed Central (PMC7613561) and its supplement;
this attempt read the same text. The paper's code is
[starquakes/mea_spineca_mapping](https://github.com/starquakes/mea_spineca_mapping), read at
commit 99c2a478393341a24c58d6ef6ab5db5b0b528d03.

**Dandiset.** [DANDI:000223](https://dandiarchive.org/dandiset/000223/0.260528.0906), version
0.260528.0906. The paper cites the dandiset without a version. It has two published versions,
0.220823.0826 (August 2022, the month the paper appeared) and 0.260528.0906 (May 2026). The data
check below shows that both list the same 20 assets with the same asset and blob identifiers, so
their files are identical, and the most recent published version is pinned, as the protocol asks
when the paper cites none.

**Method.** The pipeline of the claims file: the code's dF/F, demixing, convolution, network burst
detection and surrogates, with every parameter's source given in the helper `spineca.py`. Details
the claims file leaves open and the paper settles are cited where they are used. The claims file's
rule for the open burst parameters is followed: for each recording, the pair (p_thresh, hyperactive
strategy) whose best-R is nearest to Table 1 is carried forward to every claim.

| claim | tested here |
|---|---|
| data-scale | tested |
| table1-best-r | tested |
| fig1e-best-r-gap | tested |
| table1-optimal-thresholds | tested |
| mean-max-fscore | tested |
| acceptance-9-of-20 | tested |
| best-r-accepted-vs-rejected | tested |
| best-r-not-predictive | tested |
| fig5a-rejection-scenarios | tested |
| fig5a-threshold-order | tested |
| table1-network-measures | tested |
| fig5c-spike-contrast-vs-fscore | tested |
| fig5d-out-in-vs-fscore | tested |

In [ ]:
import io
import itertools
import time
import warnings
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

import matplotlib.pyplot as plt
import neo
import numpy as np
import pandas as pd
import quantities as pq
import requests
from elephant.spike_train_synchrony import spike_contrast
from scipy import stats
from scipy.optimize import linear_sum_assignment

import spineca as sc

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
started = time.time()

assets = sc.list_assets()
with ThreadPoolExecutor(10) as ex:
    sessions = list(ex.map(lambda a: sc.load_session(a["asset_id"]), assets))
for a, s in zip(assets, sessions):
    s["path"], s["blob"], s["size"] = a["path"], a["blob"], a["size"]
table = pd.DataFrame(sc.TABLE1, columns=sc.TABLE1_COLUMNS)
table.index = np.arange(1, len(table) + 1)
print(f"{len(sessions)} files of dandiset {sc.DANDISET_ID} version {sc.DANDISET_VERSION} loaded "
      f"in {time.time() - started:.0f} s")

## Data check

In [ ]:
earlier = sc.list_assets("0.220823.0826")
same = {(a["asset_id"], a["blob"], a["path"]) for a in earlier} == {(a["asset_id"], a["blob"], a["path"]) for a in assets}
print(f"Version 0.220823.0826 has {len(earlier)} assets, {sc.DANDISET_VERSION} has {len(assets)}; "
      f"same asset ids, blobs and paths: {same}")
info = sc.version_info()
print("Archive metadata: species", [x.get("name") for x in info["metadata"]["assetsSummary"].get("species", [])],
      "| subjects", info["metadata"]["assetsSummary"].get("numberOfSubjects"),
      "| files", info["metadata"]["assetsSummary"].get("numberOfFiles"))

files = pd.DataFrame([{
    "file": s["path"].split("/")[1].replace("_ecephys+ophys.nwb", ""), "subject": s["subject_id"],
    "start": s["session_start"][:19], "GB": round(s["size"] / 1e9, 2), "units": len(s["trains"]),
    "spikes": sum(len(t) for t in s["trains"]), "channels": s["n_channels"],
    "MEA_s": round(s["mea_duration"], 2), "frames": s["n_frames"],
    "img_Hz_meta": round(s["imaging_rate"], 2), "img_Hz_ts": round(1 / np.median(np.diff(s["frame_times"])), 2),
    "first_frame_s": round(s["frame_times"][0], 3), "last_frame_s": round(s["frame_times"][-1], 2),
    "image": "x".join(map(str, s["image_shape"][1:])), "rois": s["roi_traces"].shape[1],
    "roi_pixels": [int(m.sum()) for m in s["masks"]], "blob": s["blob"][:8]} for s in sessions])
print(files.sort_values(["subject", "start"]).to_string(index=False))

dups = files[files.duplicated("blob", keep=False)]
print("\nFiles that share a blob (the same bytes):")
print(dups[["file", "subject", "start", "units", "frames", "blob"]].to_string(index=False))
# keep one file per recording: the copy whose name carries the session
unique = [i for i, s in enumerate(sessions)
          if not (files.blob.duplicated(keep=False)[i] and "_ses-" not in s["path"])]
print(f"\nRecordings counted once: {len(unique)}; subjects: {files.loc[unique, 'subject'].nunique()}")

# other pairs with the same unit, spike and frame counts
sig = files.loc[unique].groupby(["units", "spikes", "frames"]).size()
print("Other recordings sharing unit, spike and frame counts:", int((sig > 1).sum()))

In [ ]:
rows = []
for i in unique:
    s = sessions[i]
    rates = sc.unit_rates(s)
    sorted_ok = all(np.all(np.diff(t) >= 0) for t in s["trains"])
    allsp = np.concatenate(s["trains"])
    rows.append({"file": files.file[i], "units": len(s["trains"]), "units_0_spikes": int(sum(len(t) == 0 for t in s["trains"])),
                 "rate_min_Hz": rates.min().round(3), "rate_median_Hz": np.median(rates).round(2),
                 "rate_max_Hz": rates.max().round(1), "spike_min_s": allsp.min().round(3),
                 "spike_max_s": allsp.max().round(2), "MEA_s": round(s["mea_duration"], 2),
                 "sorted": sorted_ok, "MEA_Hz": s["mea_rate"], "MEA_start": s["mea_start"],
                 "ROI_Hz": round(s["roi_rate"], 3), "indicator": s["indicator"]})
units_check = pd.DataFrame(rows)
print(units_check.to_string(index=False))
print("\nUnit table columns:", sorted({c for i in unique for c in sessions[i]["unit_cols"]}))
print("Imaging rate range over recordings (timestamps):", files.loc[unique, "img_Hz_ts"].min(), "to",
      files.loc[unique, "img_Hz_ts"].max(), "Hz; Methods: 7 to 15 Hz")
print("Recording length:", files.loc[unique, "MEA_s"].min(), "to", files.loc[unique, "MEA_s"].max(),
      "s; Methods: typically 2 to 3 min")
print("Last frame time minus MEA duration (s):",
      np.round([sessions[i]["frame_times"][-1] - sessions[i]["mea_duration"] for i in unique], 3))
print("Spine ROI not larger than its shaft ROI in every pair:",
      all(int(sessions[i]["masks"][a].sum()) <= int(sessions[i]["masks"][b].sum())
          for i in unique for _, a, b in sc.spines(sessions[i])))
dd = []
for i in unique:
    s = sessions[i]
    for c in range(s["roi_traces"].shape[1]):
        d = sc.dff(s["roi_traces"][:, c])
        dd.append(np.nanmax(np.abs(d - s["dff_stored"][:, c])))
print(f"Stored Dff against dF/F recomputed from the ROI traces (25th percentile, 100 frames): "
      f"largest absolute difference per ROI {np.min(dd):.4f} to {np.max(dd):.4f}")

The two copies of the sub-3112 recording of 2021-02-27 19:02:33 are the same blob, so the
dandiset holds 19 distinct recordings from 10 subjects, as the paper reports 19 recordings from
10 cultures; each subject id is a culture. The units table holds only spike times: no
electrode, waveform or quality columns. ROIs come in pairs, the smaller first, which fits the
code's order (spine, then its shaft; notebook 1). The stored `Dff` series is not exactly the
dF/F of the ROI traces with the paper's settings, so it is not used; dF/F is recomputed.

In [ ]:
# Clock check: the shaft trace carries the back-propagating action potentials of the imaged cell,
# which fire with the network. Its correlation with the population rate (sum of all units'
# convolved rates at the frame times) should peak near zero lag if the two streams share a clock.
lags = np.arange(-30, 31)
clock = []
for i in unique:
    s = sessions[i]
    s["rates"], s["U"] = sc.convolved_traces(s)
    pop = s["rates"].sum(axis=0)
    for k, a, b in sc.spines(s):
        sh = sc.dff(s["roi_traces"][:, b])
        r = [np.corrcoef(sh[max(L, 0):len(sh) + min(L, 0)], pop[max(-L, 0):len(pop) - max(L, 0)])[0, 1] for L in lags]
        far = [np.corrcoef(np.roll(sh, sft), pop)[0, 1] for sft in (300, 450, 600)]
        clock.append({"file": files.file[i], "spine": k, "peak_lag_frames": int(lags[np.argmax(r)]),
                      "peak_lag_s": round(lags[np.argmax(r)] / s["imaging_rate"], 2), "R_peak": round(max(r), 3),
                      "R_lag0": round(r[30], 3), "R_shift_30_to_60s_max": round(max(far), 3)})
clock = pd.DataFrame(clock)
print(clock.to_string(index=False))
print("Peak lag within 2 frames of zero:", int((clock.peak_lag_frames.abs() <= 2).sum()), "of", len(clock))

In [ ]:
# The helper computes the convolution at the frame times directly. Compare with elephant's
# instantaneous_rate at 20 kHz, sampled with np.searchsorted, as the code does, for three units.
from elephant import kernels
from elephant.statistics import instantaneous_rate

s = sessions[unique[0]]
mea_times = np.arange(s["n_mea_samples"]) / sc.MEA_FS
ds_idx = np.searchsorted(mea_times, s["frame_times"])
print("Sample index of each frame equal to the helper's:",
      np.array_equal(ds_idx, np.round(sc.frame_sample_times(s) * sc.MEA_FS).astype(int)))
kern = kernels.ExponentialKernel(sigma=0.5 * pq.s)
for u in [0, len(s["trains"]) // 2, len(s["trains"]) - 1]:
    st = neo.SpikeTrain(s["trains"][u] * pq.s, t_stop=s["mea_duration"] * pq.s, sampling_rate=sc.MEA_FS * pq.Hz)
    ifr = instantaneous_rate(st, sampling_period=st.sampling_period, center_kernel=False, kernel=kern)
    e = ifr.magnitude.squeeze()[ds_idx]
    print(f"{files.file[unique[0]]} unit {u}: {len(st)} spikes, max |elephant - helper| = "
          f"{np.max(np.abs(e - s['rates'][u])):.3f} Hz (peak {e.max():.2f} Hz), R = {np.corrcoef(e, s['rates'][u])[0, 1]:.6f}")
del mea_times

### Files and Table 1 rows

The paper gives no map from Table 1 rows to files; the claims file names culture and unit count
as the keys. Subjects are matched to cultures by their sorted unit counts, and files to rows by
unit count within a culture. Where two assignments fit equally, the one whose spike contrast
(all units, whole recording, elephant defaults) is nearer to Table 1 is taken, as an assumption,
and the other is shown. In the file with two spines, the first ROI pair is taken as Table 1's
first spine (row 9), also an assumption.

In [ ]:
# Table 1 recordings: one per row, except rows 9 and 10, two spines of one recording
rec_key = pd.Series(np.where(table.index == 10, 9, table.index), index=table.index)
rec_rows = pd.DataFrame([{"culture": table.culture[k], "units": table.units[k], "rows": list(g.index)}
                         for k, g in table.groupby(rec_key)])
subj = files.loc[unique].groupby("subject")
cult = rec_rows.groupby("culture")
S_ids, C_ids = list(subj.groups), list(cult.groups)


def group_cost(su, cu):
    a = sorted(files.loc[subj.groups[su], "units"])
    b = sorted(rec_rows.loc[cult.groups[cu], "units"])
    return sum(abs(x - y) for x, y in zip(a, b)) if len(a) == len(b) else 10 ** 6


cost = np.array([[group_cost(su, cu) for cu in C_ids] for su in S_ids])
ri, ci = linear_sum_assignment(cost)
subject_culture = {S_ids[r]: C_ids[c] for r, c in zip(ri, ci)}
ties = [(S_ids[a], S_ids[b]) for a, b in itertools.combinations(ri, 2)
        if cost[a, ci[a]] + cost[b, ci[b]] == cost[a, ci[b]] + cost[b, ci[a]]]


def spike_contrast_of(s, t0=None, t1=None, units=None):
    t0 = 0.0 if t0 is None else t0
    t1 = s["mea_duration"] if t1 is None else t1
    idx = range(len(s["trains"])) if units is None else units
    sts = [neo.SpikeTrain(s["trains"][u][(s["trains"][u] >= t0) & (s["trains"][u] <= t1)] * pq.s,
                          t_start=t0 * pq.s, t_stop=t1 * pq.s) for u in idx]
    return spike_contrast(sts)


sc_all = {i: spike_contrast_of(sessions[i]) for i in unique}


def file_rows(subject_culture):
    """Files to Table 1 recordings within each culture, by unit count; returns {file index: rows}
    and the list of tied pairs."""
    out, tied = {}, []
    for su, cu in subject_culture.items():
        fi = list(subj.groups[su])
        rr = list(cult.groups[cu])
        c = np.array([[abs(files.units[f] - rec_rows.units[r]) for r in rr] for f in fi])
        a, b = linear_sum_assignment(c)
        for x, y in zip(a, b):
            out[fi[x]] = rec_rows.rows[rr[y]]
        for x, y in itertools.combinations(range(len(a)), 2):
            if c[a[x], b[x]] + c[a[y], b[y]] == c[a[x], b[y]] + c[a[y], b[x]]:
                tied.append((fi[a[x]], fi[a[y]]))
    return out, tied


mapping, tied_files = file_rows(subject_culture)


def sc_distance(m):
    return sum(abs(sc_all[f] - table.spike_contrast[r[0]]) for f, r in m.items())


print("Subjects tied between two cultures:", ties)
print("Files tied between two rows:", [(files.file[a], files.file[b]) for a, b in tied_files])
alternatives = []
for su1, su2 in ties:  # subjects tied for two cultures: keep the order nearer in spike contrast
    alt_sc = dict(subject_culture)
    alt_sc[su1], alt_sc[su2] = alt_sc[su2], alt_sc[su1]
    alt = file_rows(alt_sc)[0]
    print(f"Swap subjects {su1}/{su2}: summed |spike contrast - Table 1| {sc_distance(mapping):.3f} as assigned, "
          f"{sc_distance(alt):.3f} swapped")
    alternatives.append(("swap subjects " + su1 + "/" + su2, (su1, su2)))
    if sc_distance(alt) < sc_distance(mapping):
        subject_culture, mapping = alt_sc, alt
for f1, f2 in tied_files:  # files tied for two rows within a culture: the same rule
    alt = dict(mapping)
    alt[f1], alt[f2] = alt[f2], alt[f1]
    print(f"Swap files {files.file[f1]}/{files.file[f2]}: {sc_distance(mapping):.3f} as assigned, "
          f"{sc_distance(alt):.3f} swapped")
    alternatives.append(("swap files " + files.file[f1] + "/" + files.file[f2], (f1, f2)))
    if sc_distance(alt) < sc_distance(mapping):
        mapping = alt
link = pd.DataFrame([{"row": r, "culture": table.culture[r], "file": files.file[f], "subject": files.subject[f],
                      "units_file": files.units[f], "units_table": table.units[r],
                      "spike_contrast_file": round(sc_all[f], 3), "spike_contrast_table": table.spike_contrast[r]}
                     for f, rs in mapping.items() for r in rs]).sort_values("row")
print(link.to_string(index=False))

# one entry per tested spine (Table 1 row)
spine_rows = {}
for f, rs in mapping.items():
    for (k, a, b), r in zip(sc.spines(sessions[f]), rs):
        spine_rows[r] = {"file": f, "k": k, "spine_roi": a, "shaft_roi": b}
print("Spines:", len(spine_rows), "| ROI pairs in the recordings:", sum(len(sc.spines(sessions[i])) for i in unique))

## Claim data-scale

> The paper made 19 paired spine Ca2+ imaging and HD-MEA recordings in 10 cultures, tested 20
> spines (one per recording, two in one recording), and the recordings have a mean of 102 ± 43
> sorted units each.

In [ ]:
n_units = files.loc[unique, "units"].to_numpy()
print("Recordings:", len(unique), "| subjects (cultures):", files.loc[unique, "subject"].nunique(),
      "| spines (ROI pairs):", sum(len(sc.spines(sessions[i])) for i in unique))
print(f"Units per recording, 19 recordings: mean {n_units.mean():.1f}, SD (ddof 0) {n_units.std():.1f}, "
      f"SD (ddof 1) {n_units.std(ddof=1):.1f}")
per_spine = np.array([files.units[spine_rows[r]['file']] for r in sorted(spine_rows)])
print(f"Units over the 20 spine rows: mean {per_spine.mean():.1f}, SD (ddof 0) {per_spine.std():.1f}")
link["units_within_10pct"] = (link.units_file - link.units_table).abs() <= 0.1 * link.units_table
nonempty = {r: sum(len(t) > 0 for t in sessions[f]["trains"]) for f, rs in mapping.items() for r in rs}
print("Rows whose unit count differs from Table 1, with the units that have at least one spike:",
      [(r, int(files.units[f]), nonempty[r], int(table.units[r])) for f, rs in mapping.items() for r in rs
       if files.units[f] != table.units[r]])
print("Per-row unit counts within 10% of Table 1:", int(link.units_within_10pct.sum()), "of", len(link),
      "| differing rows:", link.loc[link.units_file != link.units_table, ["row", "units_file", "units_table"]].to_dict("records"))

**Result: reproduced.** The dandiset holds 19 distinct recordings (20 files, two of them the same
blob) from 10 subjects, with 20 spine and shaft ROI pairs, two in one recording. Units per
recording are 101.7 ± 42.6 (ddof 0) over the 19 recordings, against 102 ± 43. Every per-row unit
count is within 10% of Table 1; rows 16 and 18 have one unit more than Table 1, and in both files
that extra unit has no spikes, so the counts of units with spikes equal Table 1's.

## Claim table1-best-r

> For each of the 20 tested spines, the best-R (the Pearson R of the demixed spine trace with the
> convolved trace of its best-matching unit, outside network bursts) has the value given in Table
> 1, from 0.148 to 0.765.

Every reading of the open burst details is run: p_thresh 96 to 99, hyperactive strategy (i)
99.75th percentile ("pc") or (ii) 15 Hz ("hz"), with the Methods' 5-point smoothing; the code's
lowess (frac 0.0002) is a variant. Units whose convolved trace is constant over the kept frames
have no R; the code's `np.argsort` would rank them first, so they are left out of the ranking
(their number is printed).

In [ ]:
readings = {}
for r, d in spine_rows.items():
    s = sessions[d["file"]]
    readings[r] = sc.best_r_readings(s, d["spine_roi"], d["shaft_roi"])
allr = pd.DataFrame([{"row": r, **{k: v for k, v in x.items() if not isinstance(v, np.ndarray)}}
                     for r, xs in readings.items() for x in xs])
allr["table"] = allr.row.map(table.best_R)
allr["match"] = (allr.best_R - allr.table).abs() <= 0.1 * allr.table
main = allr[allr.smoothing == "span5"]
print("Best-R under each reading (5-point smoothing); columns strategy and p_thresh:")
print(main.pivot_table(index="row", columns=["strategy", "p_thresh"], values="best_R").round(3)
      .assign(table=table.best_R).to_string())
print("\nUnits without an R (constant over the kept frames), largest per spine:",
      main.groupby("row").n_undefined.max().to_dict())

# carried forward: within a recording, the (p_thresh, strategy) pair nearest to Table 1
chosen = {}
for f in set(d["file"] for d in spine_rows.values()):
    rs = [r for r, d in spine_rows.items() if d["file"] == f]
    best_pair = min(itertools.product(sc.P_THRESH, sc.STRATEGIES), key=lambda ps: sum(
        abs(main[(main.row == r) & (main.p_thresh == ps[0]) & (main.strategy == ps[1])].best_R.iloc[0] - table.best_R[r]) for r in rs))
    for r in rs:
        chosen[r] = next(x for x in readings[r] if x["smoothing"] == "span5" and x["p_thresh"] == best_pair[0]
                         and x["strategy"] == best_pair[1])
summary = pd.DataFrame([{
    "row": r, "file": files.file[spine_rows[r]["file"]], "p_thresh": c["p_thresh"], "strategy": c["strategy"],
    "bursts": c["n_bursts"], "kept_frames": c["n_keep"], "of_frames": sessions[spine_rows[r]["file"]]["n_frames"],
    "best_unit": c["best"], "best_R": round(c["best_R"], 3), "table": table.best_R[r],
    "match_carried": abs(c["best_R"] - table.best_R[r]) <= 0.1 * table.best_R[r],
    "match_any": bool(main[main.row == r].match.any()),
    "R_97_pc": round(main[(main.row == r) & (main.p_thresh == 97) & (main.strategy == "pc")].best_R.iloc[0], 3),
    "R_lowess_frac_same_pair": round(allr[(allr.row == r) & (allr.smoothing == "frac") & (allr.p_thresh == c["p_thresh"])
                                          & (allr.strategy == c["strategy"])].best_R.iloc[0], 3)}
    for r, c in sorted(chosen.items())])
print("\nCarried-forward reading per spine:")
print(summary.to_string(index=False))
print("\nSpines matching Table 1 within 10% under at least one reading:", int(summary.match_any.sum()), "of 20")
print("Under the code's default pair (97, pc):",
      int(((summary.R_97_pc - summary.table).abs() <= 0.1 * summary.table).sum()), "of 20")
print("With the lowess variant (frac 0.0002), any reading:",
      int(allr[allr.smoothing == "frac"].groupby("row").match.any().sum()), "of 20")
print("Best-R range over the 20 spines (carried forward):", summary.best_R.min(), "to", summary.best_R.max())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for r in sorted(chosen):
    v = main[main.row == r].best_R
    ax.plot([table.best_R[r]] * len(v), v, ".", color="0.6")
ax.plot(summary.table, summary.best_R, "o", color="C0", label="carried forward")
ax.plot([0, 0.85], [0, 0.85], "k-", lw=0.8)
ax.plot([0, 0.85], [0, 0.85 * 1.1], "k:", lw=0.8)
ax.plot([0, 0.85], [0, 0.85 * 0.9], "k:", lw=0.8)
ax.set_xlabel("Table 1 best-R")
ax.set_ylabel("best-R here (grey: every reading)")
ax.set_xlim(0, 0.85)
ax.set_ylim(0, 0.85)
ax.legend()
print("Plotted best-R range:", round(main.best_R.min(), 3), "to", round(main.best_R.max(), 3), "(all inside the axes)")

**Result: partially reproduced (resting on an assumption).** 18 of the 20 spines have a best-R
within 10% of Table 1 under at least one reading of p_thresh and hyperactive strategy; with the
code's lowess smoothing (frac 0.0002), 19 of 20. The two that miss are row 13 (nearest 0.142
against 0.160) and row 16 (0.538 at most against 0.676). The code's default pair (97, pc) matches
14 of 20. The row-to-file assignment of rows 14, 15, 16 and 20 rests on the spike-contrast
tie-break; with subjects 2748 and 2765 the other way round, rows 16 and 20 would both miss (the
table above gives each file's values on the other row's line).

## Claim fig1e-best-r-gap

> In the example recording of Figure 1(e), the best-R shows the largest increase over the next
> ordered R value among all units.

In [ ]:
gaps = []
for r, c in sorted(chosen.items()):
    v = np.sort(c["r"][np.isfinite(c["r"])])[::-1]
    g = -np.diff(v)
    gaps.append({"row": r, "best_R": round(v[0], 3), "second_R": round(v[1], 3), "first_gap": round(g[0], 3),
                 "largest_other_gap": round(g[1:].max(), 3), "first_gap_largest": bool(g[0] > g[1:].max()),
                 "rank_of_first_gap": int(1 + np.sum(g[1:] > g[0])), "units_with_R": len(v)})
gaps = pd.DataFrame(gaps)
print(gaps.to_string(index=False))
print("Spines whose first gap is the largest:", int(gaps.first_gap_largest.sum()), "of", len(gaps))

**Result: reproduced.** With the carried-forward reading, the gap between best-R and the second
R is the largest of all consecutive gaps for 7 of the 20 spines (rows 1, 2, 3, 4, 6, 13 and 14),
so the example of Figure 1(e) can be one of them. For the other 13 it is not. The claim is about
one unnamed example, so this is a weak test.

## Noise and signal estimates

The surrogates need the SD of the spine's pixels and of background pixels "close to the spine"
(Methods 2.7). The dandiset holds the spine and shaft ROIs but no background ROI, so, as the
claims file prescribes, three background squares near each spine are placed by a rule fixed
before any R was computed (`spineca.background_squares`), and each is run. The image series is
read in a 96 by 96 pixel window around each spine, over all frames.

In [ ]:
t0 = time.time()
bounds = {r: sc.patch_bounds(sessions[d["file"]], d["spine_roi"]) for r, d in spine_rows.items()}
patches = sc.read_patches([(r, sessions[d["file"]]["asset_id"], sessions[d["file"]]["n_frames"], bounds[r])
                           for r, d in spine_rows.items()], workers=24, frames_per_block=75)
print(f"Read {len(patches)} windows in {time.time() - t0:.0f} s")
regions, estimates, rows_n = {}, {}, []
for r, d in sorted(spine_rows.items()):
    s, P = sessions[d["file"]], patches[r]
    yy, xx = np.nonzero(s["masks"][d["spine_roi"]])
    mean_trace = P[:, yy - bounds[r][0], xx - bounds[r][2]].mean(axis=1)
    mean_img = P.mean(axis=0)
    squares, cut, n_cand = sc.background_squares(s, d["spine_roi"], bounds[r], mean_img)
    regions[r] = squares
    estimates[r] = []
    for g, q in enumerate(squares):
        e = sc.noise_signal(P, s, d["spine_roi"], bounds[r], q, np.random.default_rng([223, 1, r, g]))
        estimates[r].append(e)
        rows_n.append({"row": r, "region": g, "square_rc": (int(q[0] + bounds[r][0]), int(q[1] + bounds[r][2])),
                       "dist_px": round(q[4], 1), "square_mean": round(q[3], 1), "window_p25": round(cut, 1),
                       "spine_mean": round(mean_img[yy - bounds[r][0], xx - bounds[r][2]].mean(), 1),
                       "spine_px": len(yy), "signal_amp": round(e["amp"], 2), "noise_sd": round(e["sigma"], 2),
                       "baseline": round(e["baseline"], 1),
                       "roi_trace_equal": bool(np.allclose(mean_trace, s["roi_traces"][:, d["spine_roi"]]))})
noise_table = pd.DataFrame(rows_n)
print(noise_table.to_string(index=False))
print("Spine ROI trace in the file equal to the mean of its mask pixels in every window:", noise_table.roi_trace_equal.all())
print("Image values range over the windows:", min(int(p.min()) for p in patches.values()), "to",
      max(int(p.max()) for p in patches.values()), "(uint16, no unit given)")

## Claim acceptance-9-of-20

> The best-matching unit passed the acceptance test, and a monosynaptic connection was
> accepted, for 9 of the 20 spines (Table 1 rows 1, 2, 3, 4, 6, 9, 10, 12 and 16).

1'000 GT surrogates of the best unit and 1'000 test surrogates of the second-best unit, R with
the best unit's convolved trace over the kept frames, for each of the three noise regions.
Reading (a) takes Table 1's quantiles; reading (b), the optimal quantiles found in claim
table1-optimal-thresholds, is run in that section, below.

In [ ]:
pr_main = sc.release_probabilities(np.random.default_rng([223, 0]))
print(f"Release probabilities: {len(pr_main)}, median {np.median(pr_main):.3f}, range {pr_main.min():.4f} to {pr_main.max():.3f}")
dist = {}
for r, d in sorted(spine_rows.items()):
    s, c = sessions[d["file"]], chosen[r]
    for g, e in enumerate(estimates[r]):
        dist[(r, g)] = sc.acceptance_distributions(s, s["U"], c["keep"], c["best"], c["second"], pr_main, e,
                                                   np.random.default_rng([223, 2, r, g]))


def decide(gt, te, q_gt, q_test, best_r):
    g, t = np.nanquantile(gt, q_gt), np.nanquantile(te, q_test)
    return g, t, bool(best_r > max(g, t))


acc_a = []
for (r, g), (gt, te) in dist.items():
    gthr, tthr, ok = decide(gt, te, table.GT_Q[r], table.test_Q[r], chosen[r]["best_R"])
    acc_a.append({"row": r, "region": g, "best_R": round(chosen[r]["best_R"], 3), "GT_thr": round(gthr, 3),
                  "GT_thr_table": table.GT_thr[r], "test_thr": round(tthr, 3), "test_thr_table": table.test_thr[r],
                  "accept": ok, "accept_table": table.accept[r]})
acc_a = pd.DataFrame(acc_a)
acc_a["agree"] = acc_a.accept == acc_a.accept_table
acc_a["GT_thr_10pct"] = (acc_a.GT_thr - acc_a.GT_thr_table).abs() <= 0.1 * acc_a.GT_thr_table
acc_a["test_thr_10pct"] = (acc_a.test_thr - acc_a.test_thr_table).abs() <= 0.1 * acc_a.test_thr_table
print(acc_a[acc_a.region == 0].drop(columns="region").to_string(index=False))
print("\nPer noise region, reading (a): accepted, decisions agreeing with Table 1, thresholds within 10%:")
print(acc_a.groupby("region").agg(accepted=("accept", "sum"), agree=("agree", "sum"),
                                  GT_thr_10pct=("GT_thr_10pct", "sum"), test_thr_10pct=("test_thr_10pct", "sum")).to_string())
print("\nDecision per spine and region (1 = accept):")
print(acc_a.pivot_table(index="row", columns="region", values="accept").astype(int).assign(table=table.accept.astype(int)).T.to_string())

In [ ]:
# Rows near a threshold in Table 1: decisions over five seeds, every region, reading (a)
seeds = []
for r in [x for x in [3, 6, 16, 18] if x in spine_rows]:
    d, c = spine_rows[r], chosen[r]
    s = sessions[d["file"]]
    for g, e in enumerate(estimates[r]):
        for seed in range(5):
            pr = sc.release_probabilities(np.random.default_rng([223, 5, seed]))
            gt, te = sc.acceptance_distributions(s, s["U"], c["keep"], c["best"], c["second"], pr, e,
                                                 np.random.default_rng([223, 6, r, g, seed]))
            seeds.append({"row": r, "region": g, "seed": seed, "accept": decide(gt, te, table.GT_Q[r], table.test_Q[r], c["best_R"])[2]})
seeds = pd.DataFrame(seeds)
print(seeds.groupby(["row", "region"]).accept.agg(["sum", "count"]).rename(columns={"sum": "accepted"})
      .assign(table=lambda x: [table.accept[i] for i, _ in x.index]).to_string())

# Variants on region 0: the code's stored release probabilities, and a Bernoulli draw per spike
url = ("https://raw.githubusercontent.com/starquakes/mea_spineca_mapping/"
       "99c2a478393341a24c58d6ef6ab5db5b0b528d03/notebooks/log_release_Pr.npy")
pr_stored = np.load(io.BytesIO(requests.get(url, timeout=60).content))
print(f"\nStored release probabilities: {len(pr_stored)}, median {np.median(pr_stored):.3f}")
var = []
for r, d in sorted(spine_rows.items()):
    s, c = sessions[d["file"]], chosen[r]
    for name, pr, bern in [("stored Pr", pr_stored, False), ("Bernoulli", pr_main, True)]:
        gt, te = sc.acceptance_distributions(s, s["U"], c["keep"], c["best"], c["second"], pr, estimates[r][0],
                                             np.random.default_rng([223, 7, r]), bernoulli=bern)
        var.append({"row": r, "variant": name, "accept": decide(gt, te, table.GT_Q[r], table.test_Q[r], c["best_R"])[2]})
var = pd.DataFrame(var)
var["agree"] = var.accept == var.row.map(table.accept)
print(var.groupby("variant").agg(accepted=("accept", "sum"), agree=("agree", "sum")).to_string())

## Claim table1-optimal-thresholds

> For each recording, simulated ground-truth connections of every unit give an F-score surface
> over the quantile pair (Q_test, Q_GT), whose maximum and optimal quantiles are those of Table 1
> (max_F 0.485 to 0.870).

1'000 surrogates of every unit, for each spine's noise estimate and each of its three noise
regions. Main reading: the Methods' classification of a surrogate whose best unit is not the
ground truth (FP if it passes, else TN); the code's (FN) is a variant.

In [ ]:
t0 = time.time()
grid = []
with ProcessPoolExecutor(4) as pool:
    for r, d in sorted(spine_rows.items()):
        s, c = sessions[d["file"]], chosen[r]
        for g, e in enumerate(estimates[r]):
            R = sc.r_tensor_parallel(s, s["U"], c["keep"], pr_main, e, seed=223000 + 10 * r + g, pool=pool)
            fg = sc.fscore_grid(R)
            (Fm, cm), (Fc, _) = fg["methods"], fg["code"]
            mm, om = sc.grid_optimum(Fm)
            mc, oc = sc.grid_optimum(Fc)
            i_t, i_g = np.argwhere(Fm == mm)[0]
            grid.append({"row": r, "region": g, "units": len(s["trains"]), "maxF": round(mm, 3), "table_maxF": table.max_F[r],
                         "Q_test": om[0][0], "Q_GT": om[0][1], "n_max": len(om), "table_Q_test": table.test_Q[r],
                         "table_Q_GT": table.GT_Q[r], "maxF_code": round(mc, 3), "Q_test_code": oc[0][0],
                         "Q_GT_code": oc[0][1], "all_max": om,
                         "TP": int(cm["TP"][i_t, i_g]), "FN": int(cm["FN"][i_t, i_g]), "FP": int(cm["FP"][i_t, i_g]),
                         "TN": int(cm["TN"][i_t, i_g])})
            del R
grid = pd.DataFrame(grid)
print(f"F-score grids computed in {time.time() - t0:.0f} s")
grid["maxF_10pct"] = (grid.maxF - grid.table_maxF).abs() <= 0.1 * grid.table_maxF
grid["maxF_code_10pct"] = (grid.maxF_code - grid.table_maxF).abs() <= 0.1 * grid.table_maxF
grid["Q_10pct"] = [any(abs(qt - t) <= 0.1 * t and abs(qg - gq) <= 0.1 * gq for qt, qg in m)
                   for m, t, gq in zip(grid.all_max, grid.table_Q_test, grid.table_Q_GT)]
print(grid.drop(columns="all_max").to_string(index=False))
recs = grid[grid.row != 10]  # recording 3 counted once (row 9's estimate); row 10 shown above
print("\nRecordings (19) with max F within 10% of Table 1, per noise region; Methods reading, code reading, "
      "and with Q_test and Q_GT also within 10%:")
print(recs.groupby("region").agg(maxF=("maxF_10pct", "sum"), maxF_code=("maxF_code_10pct", "sum"),
                                 quantiles=("Q_10pct", "sum")).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
for g, m in zip(range(3), "o^s"):
    x = recs[recs.region == g]
    ax.plot(x.table_maxF, x.maxF, m, label=f"region {g}, Methods", alpha=0.7)
    ax.plot(x.table_maxF, x.maxF_code, m, mfc="none", color="0.5", alpha=0.7)
ax.plot([0.3, 1], [0.3, 1], "k-", lw=0.8)
ax.set_xlabel("Table 1 max F")
ax.set_ylabel("max F here (open: code reading)")
ax.set_xlim(0.3, 1)
ax.set_ylim(0.3, 1)
ax.legend(fontsize=8)
print("Plotted max F range:", grid[["maxF", "maxF_code"]].min().min(), "to", grid[["maxF", "maxF_code"]].max().max())

**Result: partially reproduced.** Under the Methods' classification (the claims file's main
reading), the maximum F is within 10% of Table 1 for 15 of the 19 recordings with every one of
the three noise regions; it misses for rows 6, 14 and 15 (higher here) and 17 (lower). With the
code's classification (a variant), 17 of 19 match. The optimal quantiles are not those of Table
1: Q_test is 0.939 to 0.970 here for every recording, and the pair lies within 10% of Table 1's
for 0 or 1 of 19 recordings. Table 1's Q_test spans 0.545 to 0.960.

## Claim mean-max-fscore

> The simulated ground-truth test gives large F-scores across experiments, a mean maximum
> F-score of 0.692 ± 0.136 over 19 recordings with 1'000 surrogates per unit.

In [ ]:
mf = []
for g in range(3):
    for col, name in [("maxF", "Methods"), ("maxF_code", "code")]:
        x19 = recs[recs.region == g][col].to_numpy()
        x20 = grid[grid.region == g][col].to_numpy()
        x19b = grid[(grid.region == g) & (grid.row != 9)][col].to_numpy()
        mf.append({"region": g, "reading": name, "mean_19": round(x19.mean(), 3), "sd_19": round(x19.std(), 3),
                   "mean_19_row10": round(x19b.mean(), 3), "sd_19_row10": round(x19b.std(), 3),
                   "mean_20": round(x20.mean(), 3), "sd_20": round(x20.std(), 3)})
mf = pd.DataFrame(mf)
mf["match"] = ((mf.mean_19 - 0.692).abs() <= 0.0692) & ((mf.sd_19 - 0.136).abs() <= 0.0136)
print(mf.to_string(index=False))
print("Table 1: mean 0.692, SD 0.136 over 19 recordings (ddof 0)")

**Result: partially reproduced.** Under the Methods' classification the mean maximum F over 19
recordings is 0.695 (expected 0.692 ± 0.069) and the SD 0.121 (ddof 0), just below the tolerance
of 0.136 ± 0.0136, with every noise region (0.120 to 0.121). With the code's classification the mean
is 0.666 and the SD 0.133, both within tolerance. The F-scores are large across recordings, as the
paper says; the spread under the main reading is slightly smaller.

### Claim acceptance-9-of-20, reading (b): the optimal quantiles found here

In [ ]:
acc_b = []
for (r, g), (gt, te) in dist.items():
    gr = grid[(grid.row == r) & (grid.region == g)].iloc[0]
    decisions = [decide(gt, te, qg, qt, chosen[r]["best_R"]) for qt, qg in gr.all_max]
    gthr, tthr, ok = decisions[0]
    acc_b.append({"row": r, "region": g, "Q_test": gr.Q_test, "Q_GT": gr.Q_GT, "best_R": round(chosen[r]["best_R"], 3),
                  "GT_thr": round(gthr, 3), "GT_thr_table": table.GT_thr[r], "test_thr": round(tthr, 3),
                  "test_thr_table": table.test_thr[r], "accept": ok, "accept_table": table.accept[r],
                  "same_decision_all_maxima": len({x[2] for x in decisions}) == 1})
acc_b = pd.DataFrame(acc_b)
acc_b["agree"] = acc_b.accept == acc_b.accept_table
print(acc_b[acc_b.region == 0].drop(columns="region").to_string(index=False))
print("\nPer noise region, reading (b): accepted, decisions agreeing with Table 1:")
print(acc_b.groupby("region").agg(accepted=("accept", "sum"), agree=("agree", "sum"),
                                  stable=("same_decision_all_maxima", "all")).to_string())
print("\nDecision per spine and region (1 = accept):")
print(acc_b.pivot_table(index="row", columns="region", values="accept").astype(int).assign(table=table.accept.astype(int)).T.to_string())

**Result: partially reproduced.** Every reading and noise region gives 17 or 18 decisions that
agree with Table 1, the claims file's band for a partial reproduction, so the outcome does not
depend on the noise region. With Table 1's quantiles, 7 spines are accepted with each region, and
18 decisions agree; rows 3 and 16, accepted in Table 1, are rejected here, in each of five seeds.
Row 3 has the Table 1 best-R (0.222) but a test threshold of 0.251 against 0.190; row 16's best-R
is 0.538, not 0.676, and its GT threshold 0.684. With the optimal quantiles found here, 7, 7 and 6
are accepted (18, 18 and 17 agree). The stored release probabilities and a Bernoulli draw per
spike give the same 7 accepted. GT thresholds are within 10% of Table 1 for 12 or 13 spines and
test thresholds for 10 or 11. Rows 9 and 10, which Table 1 gives identical thresholds, have
different ones here (each spine has its own noise estimate).

## Claim best-r-accepted-vs-rejected

> Accepted connections have larger best-R values than rejected ones (0.584 ± 0.149 against 0.331
> ± 0.124; Mann-Whitney U = 12.0, n1 = 9, n2 = 11, P < 0.05).

In [ ]:
mw = []
for name, acc in [("a", acc_a), ("b", acc_b)]:
    for g in range(3):
        x = acc[acc.region == g]
        A, B = x[x.accept].best_R.to_numpy(), x[~x.accept].best_R.to_numpy()
        if len(A) and len(B):
            u = stats.mannwhitneyu(A, B, alternative="two-sided")
            u1 = stats.mannwhitneyu(A, B, alternative="greater")
            mw.append({"reading": name, "region": g, "n_acc": len(A), "n_rej": len(B), "mean_acc": round(A.mean(), 3),
                       "sd_acc": round(A.std(), 3), "mean_rej": round(B.mean(), 3), "sd_rej": round(B.std(), 3),
                       "U_acc": u.statistic, "U_min": min(u.statistic, len(A) * len(B) - u.statistic),
                       "p_two_sided": round(u.pvalue, 4), "p_one_sided": round(u1.pvalue, 4)})
        else:
            mw.append({"reading": name, "region": g, "n_acc": len(A), "n_rej": len(B)})
mw = pd.DataFrame(mw)
print(mw.to_string(index=False))
print("Paper: 0.584 ± 0.149 against 0.331 ± 0.124, U = 12.0, n1 = 9, n2 = 11, P < 0.05")

**Result: partially reproduced.** With this attempt's classification (7 accepted, 13 rejected,
reading (a), any noise region), accepted spines have larger best-R: 0.622 ± 0.081 against 0.337 ±
0.137, Mann-Whitney U = 5 (smaller U; 86 for the accepted group), two-sided p = 0.0015. Both means
are within tolerance of 0.584 and 0.331 and the test gives P < 0.05 with the accepted group higher,
but U is 5, not 12 ± 1.2; with 7 and 13 spines in place of 9 and 11 the U values are not
comparable. Region 2 with the optimal quantiles gives U = 10 (6 and 14 spines).

## Claim best-r-not-predictive

> The best-R alone does not predict the outcome of the acceptance test; connections with R as
> high as about 0.6 were rejected, while an R of about 0.2 was enough for acceptance in another
> case.

In [ ]:
npred = []
for name, acc in [("a", acc_a), ("b", acc_b)]:
    for g in range(3):
        x = acc[acc.region == g]
        mx_rej = x[~x.accept].best_R.max() if (~x.accept).any() else np.nan
        mn_acc = x[x.accept].best_R.min() if x.accept.any() else np.nan
        npred.append({"reading": name, "region": g, "max_rejected": mx_rej, "row_max_rejected": x[~x.accept].row.iloc[x[~x.accept].best_R.argmax()] if (~x.accept).any() else None,
                      "min_accepted": mn_acc, "row_min_accepted": x[x.accept].row.iloc[x[x.accept].best_R.argmin()] if x.accept.any() else None,
                      "overlap": bool(mx_rej > mn_acc)})
npred = pd.DataFrame(npred)
npred["max_rej_within"] = (npred.max_rejected - 0.6).abs() <= 0.06
npred["min_acc_within"] = (npred.min_accepted - 0.2).abs() <= 0.02
print(npred.to_string(index=False))

**Result: partially reproduced.** The largest best-R of a rejected spine is 0.625 (row 17),
within tolerance of about 0.6, and it exceeds the smallest accepted best-R, so no cut on best-R
separates the groups. The smallest accepted best-R is 0.515 (row 2), not about 0.2: row 3, the
accepted spine with R near 0.2 in Table 1, is rejected here.

## Claims fig5a-rejection-scenarios and fig5a-threshold-order

> Connections were rejected in three scenarios, best-R below the test threshold only, below the
> GT threshold only, or below both, which shows the advantage of having two thresholds.

> For some recordings the test threshold is larger than the GT threshold, and for others the
> reverse.

In [ ]:
sc_rows = []
for name, acc in [("a", acc_a), ("b", acc_b)]:
    for g in range(3):
        x = acc[acc.region == g]
        rej = x[~x.accept]
        sc_rows.append({"reading": name, "region": g, "rejected": len(rej),
                        "below_test_only": int(((rej.best_R <= rej.test_thr) & (rej.best_R > rej.GT_thr)).sum()),
                        "below_GT_only": int(((rej.best_R <= rej.GT_thr) & (rej.best_R > rej.test_thr)).sum()),
                        "below_both": int(((rej.best_R <= rej.GT_thr) & (rej.best_R <= rej.test_thr)).sum()),
                        "test_above_GT": int((x.test_thr > x.GT_thr).sum()), "GT_above_test": int((x.GT_thr > x.test_thr).sum())})
print(pd.DataFrame(sc_rows).to_string(index=False))
print("Table 1: 4 below test only, 1 below GT only, 6 below both; test above GT for 15 spines, below for 5")

**Result: both claims reproduced.** All three rejection scenarios occur among the rejected spines
under every reading and region: with Table 1's quantiles and region 0, 5 below the test threshold
only, 4 below the GT threshold only and 4 below both (Table 1: 4, 1 and 6). The test threshold is
above the GT threshold for 14 spines and below it for 6 (Table 1: 15 and 5); the other readings
give 15 to 18 and 2 to 5.

## Claim table1-network-measures

> Each recording's spike contrast (0.17 to 0.69) and ratio of spikes outside to inside network
> bursts (0.060 to 2.003) have the values given in Table 1.

Every combination of the open details: (a) all sorted units or those kept after the hyperactive
exclusion, (b) for OUT/IN, burst periods with or without the extensions, (c) the whole MEA
recording or the span of the imaging frames. The bursts are those of the carried-forward
reading.

In [ ]:
net = []
for f, rs in mapping.items():
    s = sessions[f]
    c = chosen[rs[0]]
    drop = set(sc.hyperactive(s, c["strategy"]).tolist())
    for units_set in ["all", "clean"]:
        idx = list(range(len(s["trains"]))) if units_set == "all" else [u for u in range(len(s["trains"])) if u not in drop]
        for span in ["whole", "imaging"]:
            t0_, t1_ = (0.0, s["mea_duration"]) if span == "whole" else (s["frame_times"][0], s["frame_times"][-1])
            scv = spike_contrast_of(s, t0_, t1_, idx)
            for ext in ["extended", "detected"]:
                a, b = (c["starts"] - 0.1, c["ends"] + 0.5) if ext == "extended" else (c["starts"], c["ends"])
                oi = sc.out_in_ratio([s["trains"][u] for u in idx], a, b, t0_, t1_)
                net.append({"row": rs[0], "units": units_set, "span": span, "bursts": ext, "spike_contrast": scv,
                            "out_in": oi, "sc_table": table.spike_contrast[rs[0]], "oi_table": table.out_in[rs[0]]})
net = pd.DataFrame(net)
net["sc_ok"] = (net.spike_contrast - net.sc_table).abs() <= 0.1 * net.sc_table
net["oi_ok"] = (net.out_in - net.oi_table).abs() <= 0.1 * net.oi_table
net["both"] = net.sc_ok & net.oi_ok
combo = net.groupby(["units", "span", "bursts"]).agg(sc_match=("sc_ok", "sum"), oi_match=("oi_ok", "sum"),
                                                      both_match=("both", "sum"), recordings=("row", "count"))
print(combo.to_string())
best_combo = combo.both_match.idxmax()
print("\nCombination with most recordings matching both:", best_combo)
bc = net[(net.units == best_combo[0]) & (net.span == best_combo[1]) & (net.bursts == best_combo[2])].sort_values("row")
print(bc[["row", "spike_contrast", "sc_table", "sc_ok", "out_in", "oi_table", "oi_ok"]].round(3).to_string(index=False))
print("Spike contrast range:", round(bc.spike_contrast.min(), 3), "to", round(bc.spike_contrast.max(), 3),
      "| OUT/IN range:", round(bc.out_in.min(), 3), "to", round(bc.out_in.max(), 3))

# Variant: OUT/IN with the bursts of every reading of p_thresh and strategy (5-point smoothing),
# all units, whole recording, extended bursts
oi_var = []
for f, rs in mapping.items():
    s = sessions[f]
    for p_, st_ in itertools.product(sc.P_THRESH, sc.STRATEGIES):
        a, b, _ = sc.detect_bursts(s, p_, st_, "span5")
        oi_var.append({"row": rs[0], "reading": f"{p_}{st_}", "out_in": sc.out_in_ratio(s["trains"], a - 0.1, b + 0.5, 0.0, s["mea_duration"])})
oi_var = pd.DataFrame(oi_var).pivot_table(index="row", columns="reading", values="out_in")
oi_var["table"] = [table.out_in[r] for r in oi_var.index]
oi_var["any_within_10pct"] = [any(abs(oi_var.loc[r, c] - oi_var.table[r]) <= 0.1 * oi_var.table[r]
                                  for c in oi_var.columns[:-1]) for r in oi_var.index]
print("\nVariant, OUT/IN under each burst reading:")
print(oi_var.round(3).to_string())
print("Recordings with OUT/IN within 10% of Table 1 under some burst reading:", int(oi_var.any_within_10pct.sum()), "of", len(oi_var))

**Result: not reproduced.** Spike contrast matches Table 1 within 10% for all 19 recordings when
all sorted units are used (16 of 19 without the hyperactive units). OUT/IN matches for at most 6 of
19 recordings under any combination (all units, extended bursts), so both measures match for 6, below
the 10 needed for a partial reproduction. OUT/IN here is mostly higher than Table 1's (for row 8,
0.419 against 0.060). With the bursts of any p_thresh and strategy, 10 of 19 recordings can be
matched one by one, but no single reading matches them all. The spike contrast part holds; the
OUT/IN values of Table 1 could not be recovered with the burst detection described.

## Claims fig5c-spike-contrast-vs-fscore and fig5d-out-in-vs-fscore

> The network's spike contrast is negatively correlated with the maximum F-score of the
> recording (Spearman R = -0.737, P = 0.0002), so more synchronous networks give poorer
> classification.

> The ratio of spikes outside to inside network bursts is positively correlated with the maximum
> F-score of the recording (Spearman R = 0.672, P = 0.0012).

In [ ]:
rho = []
for g in range(3):
    for col in ["maxF", "maxF_code"]:
        for combo_name, sub in net.groupby(["units", "span", "bursts"]):
            m20 = grid[grid.region == g].set_index("row")[col]
            meas = sub.set_index("row")
            rows20 = sorted(spine_rows)
            x_sc = [meas.spike_contrast[r if r != 10 else 9] for r in rows20]
            x_oi = [meas.out_in[r if r != 10 else 9] for r in rows20]
            y = [m20[r] for r in rows20]
            rows19 = [r for r in rows20 if r != 10]
            i19 = [rows20.index(r) for r in rows19]
            for nm, x in [("spike_contrast", x_sc), ("out_in", x_oi)]:
                r20 = stats.spearmanr(x, y)
                r19 = stats.spearmanr(np.array(x)[i19], np.array(y)[i19])
                rho.append({"region": g, "F": col, "units": combo_name[0], "span": combo_name[1], "bursts": combo_name[2],
                            "measure": nm, "R_20": round(r20.statistic, 3), "P_20": r20.pvalue,
                            "R_19": round(r19.statistic, 3), "P_19": r19.pvalue})
rho = pd.DataFrame(rho)
main_rho = rho[(rho.units == best_combo[0]) & (rho.span == best_combo[1]) & (rho.bursts == best_combo[2])]
print("Combination of claim table1-network-measures:", best_combo)
print(main_rho.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("\nOver every combination, region and F reading: R range")
print(rho.groupby("measure")[["R_20", "R_19"]].agg(["min", "max"]).to_string())
print("Largest P over every combination, region and F reading:")
print(rho.groupby("measure")[["P_20", "P_19"]].max().to_string(float_format=lambda v: f"{v:.4g}"))
print("Paper: spike contrast R = -0.737 (P = 0.0002), OUT/IN R = 0.672 (P = 0.0012)")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 4))
g0 = grid[grid.region == 0].set_index("row")
for ax, nm, lab in [(axs[0], "spike_contrast", "spike contrast"), (axs[1], "out_in", "OUT/IN ratio")]:
    xx = [bc.set_index("row")[nm][r if r != 10 else 9] for r in sorted(spine_rows)]
    ax.plot(xx, [g0.maxF[r] for r in sorted(spine_rows)], "o", label="here (region 0)")
    ax.plot([table[nm][r] for r in sorted(spine_rows)], [table.max_F[r] for r in sorted(spine_rows)], "x", color="0.5", label="Table 1")
    ax.set_xlabel(lab)
    ax.set_ylabel("max F")
    ax.legend(fontsize=8)
print("Plotted: all points inside the automatic axes")

**Result: fig5c-spike-contrast-vs-fscore reproduced; fig5d-out-in-vs-fscore partially reproduced
(resting on an assumption).** Over the 20 rows, spike contrast against maximum F gives Spearman
R = -0.724 (P = 0.0003) with noise region 0 and the Methods' classification, and -0.724 to -0.781
under every combination, region and classification, all within -0.737 ± 0.074. OUT/IN against
maximum F gives R = 0.592 (P = 0.006; 0.598 over 19 recordings), below 0.672 ± 0.067, with region 0,
the background square nearest the spine, taken as the Methods' "background ROI close to the spine".
Region 1 gives 0.615 and the code's classification 0.646, both within tolerance; over every
combination R is 0.592 to 0.759, always positive with P below 0.05. The OUT/IN values themselves
differ from Table 1 (claim table1-network-measures).

## Summary

| claim | outcome |
|---|---|
| data-scale | reproduced |
| table1-best-r | partially reproduced (assumption: row-to-file assignment) |
| fig1e-best-r-gap | reproduced |
| table1-optimal-thresholds | partially reproduced |
| mean-max-fscore | partially reproduced |
| acceptance-9-of-20 | partially reproduced |
| best-r-accepted-vs-rejected | partially reproduced |
| best-r-not-predictive | partially reproduced |
| fig5a-rejection-scenarios | reproduced |
| fig5a-threshold-order | reproduced |
| table1-network-measures | not reproduced |
| fig5c-spike-contrast-vs-fscore | reproduced |
| fig5d-out-in-vs-fscore | partially reproduced (assumption: noise region) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The convolution is computed at the frame times,
not at every 20 kHz sample (the largest difference from elephant is 0.3 Hz on peaks of 19 to 45
Hz). The paper chose p_thresh by eye for each recording and does not report it; the claims file's
rule picks the reading nearest to Table 1's best-R. The background region for the noise estimate
is not in the dataset; three squares near each spine are placed by a fixed rule, and region 0,
the nearest, is used where a choice matters. Units with an undefined R are left out of the
ranking. The release probabilities are a fresh draw. Table 1 rows are matched to files by
culture and unit count, with ties broken by spike contrast.

**What was found about the dataset.** The 20 files hold 19 recordings: sub-3112_ecephys+ophys.nwb
is the same blob as sub-3112_ses-20210227T190233. Nothing maps files to the paper's Table 1 rows
or cultures, and two pairs of recordings cannot be told apart by unit count. The background ROIs
the paper used for the noise estimate are not in the files. The ROIs are not labelled as spine or
shaft. The imaging plane's indicator is "unknown", where the paper used jGCaMP7b. The stored `Dff`
series is not the dF/F the paper describes (differences up to 0.23). Two files each hold one
unit with no spikes, which Table 1 does not count. The two streams share a clock: the shaft trace
correlates best with the population rate at zero lag in 19 of 20 spines and at one frame in the
other.

In [ ]:
print(f"Notebook run time: {(time.time() - started) / 60:.1f} min")